# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pankaj1281/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
%pip -q install duckdb huggingface_hub


In [14]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

In [15]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [26]:
con.sql(f"""
SELECT COUNT(*) AS rows
FROM {TABLES["fact_daily"]}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows
0,78835655


In [17]:
con.sql(f"""
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES["fact_daily"]}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date
0,2025-01-27,2026-06-30


In [18]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS c
FROM {TABLES["fact_daily"]}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c
0,2026-06-16,client_9c26c096d6e57253,content_70b02f5039a882c2,2
1,2026-06-18,client_810019792c9b8efc,content_fa5355f17fb01178,2
2,2026-06-18,client_b77d0d5f08f05e64,content_6b6a532cb09428b2,2
3,2026-06-17,client_1a8bf67cad4ee525,content_dd441f559f3cb4f7,2
4,2026-06-17,client_1a8bf67cad4ee525,content_10c2a0e85769f485,2


In [19]:
march = con.sql(f"""
SELECT *
FROM {TABLES["fact_daily"]}
WHERE date_trunc('month', report_date) = DATE '2026-03-01'
LIMIT 10
""").df()

march.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## Unit of analysis

One row represents the daily search performance of one content item for one client on one report date.

Grain:
report_date × client_hash_id × content_hash_id

Time window:
The warehouse contains data from 2025-01-27 to 2026-06-30.
For this assignment, analysis is performed on the March 2026 partition.

In [20]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES["fact_daily"]}
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,78835655,2025-01-27,2026-06-30


In [27]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicates
FROM {TABLES["fact_daily"]}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicates
0,2026-06-19,client_06d356715a8ff3b6,content_6c6a2658f025ec02,2
1,2026-06-22,client_8ddc46da5414ffd8,content_55335cfcf3499724,2
2,2026-06-23,client_1a8bf67cad4ee525,content_e85eff1aad797f4c,2
3,2026-06-23,client_1a8bf67cad4ee525,content_2630830d5f397c6c,2
4,2026-06-23,client_1a8bf67cad4ee525,content_fb1b290d16ca0d29,2


## Feature
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engagement_rate

## Label
- is_declining
- trend_direction
- trend_pct

## Context
- client_hash_id
- content_hash_id
- report_date

## Excluded
- trend_direction (used to create the label)
- trend_pct (used to create the label)
- IDs (identifiers only)

In [21]:
con.sql(f"""
DESCRIBE
SELECT *
FROM {TABLES["fact_daily"]}
""").df()


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


The contract is verified using:

- Row count
- Date range
- Grain check
- Missing value check

In [22]:
con.sql(f"""
SELECT COUNT(*) AS rows
FROM {TABLES["fact_daily"]}
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows
0,78835655


In [23]:
con.sql(f"""
SELECT
AVG(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions,
AVG(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_clicks,
AVG(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_position
FROM {TABLES["fact_daily"]}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,missing_impressions,missing_clicks,missing_position
0,0.001243,0.001243,0.632527


In [24]:
con.sql(f"""
SELECT
MIN(report_date),
MAX(report_date)
FROM {TABLES["fact_daily"]}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min(report_date),max(report_date)
0,2025-01-27,2026-06-30


## Data limitations

- Clients have different history lengths.
- Early rows may not contain GA4 data.
- GA4 zero values before data availability do not necessarily indicate zero engagement.
- Query windows overlap and can cause label leakage if used incorrectly.
- IDs cannot be used as model features.

In [25]:
con.sql(f"""
SELECT
client_hash_id,
MIN(report_date) AS first_day,
MAX(report_date) AS last_day,
COUNT(*) AS rows
FROM {TABLES["fact_daily"]}
GROUP BY client_hash_id
ORDER BY first_day
LIMIT 10
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,first_day,last_day,rows
0,client_9958f0a7ae1df715,2025-01-27,2026-06-30,1470881
1,client_ff644d8251367cbb,2025-01-27,2026-06-30,1246888
2,client_73cda7b4e4f265ea,2025-02-11,2026-06-30,8708971
3,client_fef1a8f436438636,2025-03-11,2026-06-30,2779933
4,client_62f4a7e64f5e0096,2025-06-07,2026-06-30,5676451
5,client_b10cb2997d0c7c86,2025-06-18,2026-06-30,1050146
6,client_65de48885f4ef01b,2025-06-21,2026-06-30,3296200
7,client_c182d11e4862a37d,2025-06-21,2026-06-30,282410
8,client_3197e6291363b4db,2025-06-29,2026-06-30,2598532
9,client_625b6439094e23e4,2025-07-01,2026-06-30,7589247


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.